# Part B proof-of-concept starter

A complete, leakage-free pipeline for our proof of concept: **data index, split by object/trial, baseline, model, validation-only model selection, test metrics, failure cases and a demo clip.**

**How to use it**
1. **Now:** *Runtime > Run all* with `DEMO_MODE = True`. It builds a small synthetic dataset and runs end to end in about a minute, so we know the pipeline works before our real data arrives.
2. **Data lead:** write `load_index()` for our dataset (Step 1), then set `DEMO_MODE = False`.
3. **Modelling lead:** choose settings on the **validation** split only. The test split is used once, in Step 5.
4. **For the code zip:** set `TRAIN = False` so the notebook loads the saved weights and reproduces every number.

Works for classification (accuracy, macro F1, confusion matrix) and regression (RMSE, MAE, predicted versus true), with one frame or several frames per sample (`N_FRAMES`, for slip or grasp-outcome tasks).

*Part B only: never copy it into Part A.*

In [ ]:
# ---- Configuration: the only cell most people need to edit ----
DEMO_MODE  = True              # True: synthetic data to test the pipeline. False: our real dataset.
TRAIN      = True              # False: skip training and load saved weights (required for the code zip)
TASK       = "classification"  # "classification" or "regression"
N_FRAMES   = 1                 # frames per sample (e.g. 2 for before/after a grasp), stacked as channels
IMG_SIZE   = 128               # images are resized to IMG_SIZE x IMG_SIZE
MODEL_NAME = "small_cnn"       # "small_cnn" or "resnet18"
PRETRAINED = False             # resnet18 only: ImageNet weights (allowed; cite them if used)
EPOCHS     = 15
BATCH_SIZE = 32
LR         = 1e-3
PATIENCE   = 5                 # early stopping on the validation score
SEED       = 0
OUT_DIR    = "outputs"         # on Colab, point this at Google Drive so a disconnect loses nothing

In [ ]:
import json, random, time
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# On Colab, keep checkpoints safe across disconnects:
# from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/partb_outputs"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
print("device:", device, "| torch", torch.__version__)

## Step 1: the dataset index (data lead)

Everything downstream reads one table with **one row per sample**:

| column | meaning |
|---|---|
| `paths` | list of `N_FRAMES` image file paths, in time order |
| `label` | class name (classification), or a number or list of numbers (regression) |
| `group` | the object, trial or session the sample came from: **the unit we split by** |

Replace the end of `load_index()` with code that builds this table for our dataset. The demo dataset shows the idea: each synthetic "object" has its own orientation and colour, so frames from one object look alike, which is exactly why we never split by single frame.

In [ ]:
def make_demo_dataset(root="demo_data", n_classes=4, objects_per_class=8, frames_per_object=12, size=96):
    """Synthetic stripe 'textures' whose frequency depends on the class."""
    root, rows, rng = Path(root), [], np.random.default_rng(SEED)
    yy, xx = np.mgrid[0:size, 0:size] / size
    for c in range(n_classes):
        for o in range(objects_per_class):
            angle, phase, tint = rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi), rng.uniform(0.6, 1.0, 3)
            obj = f"class{c}_obj{o}"; (root / obj).mkdir(parents=True, exist_ok=True)
            for f in range(frames_per_object):
                freq = 3 + 3 * c + rng.normal(0, 0.6)
                t = xx * np.cos(angle) + yy * np.sin(angle)
                img = 0.5 + 0.5 * np.sin(2 * np.pi * freq * t + phase + 0.3 * f)
                img = np.clip(img[..., None] * tint + rng.normal(0, 0.15, (size, size, 3)), 0, 1)
                p = root / obj / f"f{f:02d}.png"
                Image.fromarray((img * 255).astype(np.uint8)).save(p)
                rows.append({"path": str(p), "label": f"texture_{c}", "freq": float(freq), "group": obj, "frame": f})
    return pd.DataFrame(rows)

def to_samples(frames, n_frames, target_col):
    """Group consecutive frames of one object or trial into samples of n_frames."""
    samples = []
    for g, d in frames.sort_values(["group", "frame"]).groupby("group"):
        paths, targets = d["path"].tolist(), d[target_col].tolist()
        for i in range(0, len(paths) - n_frames + 1, n_frames):
            samples.append({"paths": paths[i:i + n_frames], "label": targets[i + n_frames - 1], "group": g})
    return pd.DataFrame(samples)

def load_index():
    if DEMO_MODE:
        frames = make_demo_dataset()
        return to_samples(frames, N_FRAMES, "label" if TASK == "classification" else "freq")
    # ---- Data lead: build the index for our dataset here ----
    # Return a DataFrame with one row per sample and the columns paths, label, group (see above).
    raise NotImplementedError("Write load_index() for our dataset, then set DEMO_MODE = False")

index = load_index()
print(len(index), "samples from", index["group"].nunique(), "groups")
index.head()

In [ ]:
fmt = lambda v: str(v)[:18] if TASK == "classification" else np.array2string(np.atleast_1d(v), precision=2)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, (_, r) in zip(axes, index.sample(5, random_state=SEED).iterrows()):
    ax.imshow(Image.open(r["paths"][-1]).convert("RGB")); ax.set_title(fmt(r["label"]), fontsize=9); ax.axis("off")
plt.suptitle("Sample images"); plt.tight_layout(); plt.show()

## Step 2: leakage-free 60/20/20 split

Every group (object, trial or session) lands in exactly one split, and the cell **stops with an error** if any group leaks. For classification the split is also stratified, so each split keeps roughly the same class balance.

In [ ]:
def split_by_group(df):
    if TASK == "classification":
        rest_idx, test_idx = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(df, df["label"], df["group"]))
        rest = df.iloc[rest_idx]
        tr, va = next(StratifiedGroupKFold(4, shuffle=True, random_state=SEED).split(rest, rest["label"], rest["group"]))
    else:
        rest_idx, test_idx = next(GroupKFold(5).split(df, groups=df["group"]))
        rest = df.iloc[rest_idx]
        tr, va = next(GroupKFold(4).split(rest, groups=rest["group"]))
    return rest.iloc[tr].reset_index(drop=True), rest.iloc[va].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

train_df, val_df, test_df = split_by_group(index)
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert not set(a["group"]) & set(b["group"]), "Leakage: a group appears in two splits"
splits = [("train", train_df), ("val", val_df), ("test", test_df)]
print(pd.DataFrame({s: {"samples": len(d), "groups": d["group"].nunique()} for s, d in splits}).T)
if TASK == "classification":
    print(pd.DataFrame({s: d["label"].value_counts() for s, d in splits}).fillna(0).astype(int))

## Step 3: data loaders, normalised with training statistics only

Pixel means and standard deviations, and regression targets, are standardised with **training-split statistics only**. Augmentation is applied to the training split only, and only in physically valid ways.

In [ ]:
if TASK == "classification":
    classes = sorted(index["label"].unique()); class_to_idx = {c: i for i, c in enumerate(classes)}; n_out = len(classes)
    encode = lambda y: class_to_idx[y]
else:
    y_train = np.stack([np.atleast_1d(np.asarray(v, dtype=np.float32)) for v in train_df["label"]])
    y_mean, y_std = y_train.mean(0), y_train.std(0) + 1e-8          # training statistics only
    n_out = y_train.shape[1]
    encode = lambda y: (np.atleast_1d(np.asarray(y, dtype=np.float32)) - y_mean) / y_std

def load_frames(paths):
    arrs = [np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255 for p in paths]
    return np.concatenate(arrs, axis=2).transpose(2, 0, 1)          # (3 * N_FRAMES, H, W)

_stack = np.stack([load_frames(p) for p in train_df.sample(min(len(train_df), 300), random_state=SEED)["paths"]])
ch_mean = torch.tensor(_stack.mean((0, 2, 3))).view(-1, 1, 1)
ch_std = torch.tensor(_stack.std((0, 2, 3)) + 1e-6).view(-1, 1, 1)

class TactileDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df, self.augment = df.reset_index(drop=True), augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = (torch.from_numpy(load_frames(r["paths"])) - ch_mean) / ch_std
        if self.augment:
            # Physically valid only. A flip or rotation changes the direction of shear forces or slip,
            # so for those tasks the label must change too, or the augmentation must be left out.
            x = x + 0.02 * torch.randn_like(x)
        y = encode(r["label"])
        return x, torch.tensor(y, dtype=torch.long if TASK == "classification" else torch.float32)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(TactileDataset(train_df, augment=True), batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader = DataLoader(TactileDataset(val_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(TactileDataset(test_df), batch_size=BATCH_SIZE)
xb, yb = next(iter(train_loader))
print("one batch:", tuple(xb.shape), "targets", tuple(yb.shape))

## Step 4: the model

`small_cnn` trains from scratch in minutes. `resnet18` can start from ImageNet weights (`PRETRAINED = True`, cite them). With several frames, they are stacked as input channels.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_ch, n_out):
        super().__init__()
        block = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(in_ch, 32), block(32, 64), block(64, 128), block(128, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_out))
    def forward(self, x):
        return self.head(self.features(x))

def make_model():
    in_ch = 3 * N_FRAMES
    if MODEL_NAME == "small_cnn":
        return SmallCNN(in_ch, n_out)
    from torchvision.models import resnet18, ResNet18_Weights
    m = resnet18(weights=ResNet18_Weights.DEFAULT if PRETRAINED else None)
    if in_ch != 3:      # stacked frames need a new first layer (its pretrained weights are lost)
        m.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
    m.fc = nn.Linear(m.fc.in_features, n_out)
    return m

model = make_model().to(device)
print(MODEL_NAME, f"{sum(p.numel() for p in model.parameters()):,} parameters")

## Step 5: train, choosing the best epoch on the validation split

The best epoch by validation score (macro F1, or minus RMSE) is saved to `OUT_DIR/best_model.pt`. With `TRAIN = False` this cell only loads that file.

In [ ]:
loss_fn = nn.CrossEntropyLoss() if TASK == "classification" else nn.MSELoss()

def run_epoch(loader, train):
    model.train(train); total, preds, targets = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = loss_fn(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(x); preds.append(out.detach().cpu()); targets.append(y.cpu())
    return total / len(loader.dataset), torch.cat(preds), torch.cat(targets)

def val_score(preds, targets):
    """Higher is better: macro F1 (classification) or minus RMSE in standardised units (regression)."""
    if TASK == "classification":
        return f1_score(targets, preds.argmax(1), average="macro")
    return -float(torch.sqrt(((preds - targets) ** 2).mean()))

ckpt_path = out_dir / "best_model.pt"
if TRAIN:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    history, best, wait = [], -np.inf, 0
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, _, _ = run_epoch(train_loader, True)
        va_loss, vp, vt = run_epoch(val_loader, False)
        score = val_score(vp, vt); history.append((epoch, tr_loss, va_loss, score))
        print(f"epoch {epoch:2d}  train loss {tr_loss:.3f}  val loss {va_loss:.3f}  val score {score:.3f}  ({time.time() - t0:.0f}s)")
        if score > best:
            best, wait = score, 0
            torch.save({"state_dict": model.state_dict(),
                        "config": {"TASK": TASK, "N_FRAMES": N_FRAMES, "IMG_SIZE": IMG_SIZE, "MODEL_NAME": MODEL_NAME}}, ckpt_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("early stop"); break
    h = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "val_score"]); h.to_csv(out_dir / "history.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="validation")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
    ax[1].plot(h.epoch, h.val_score); ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("validation macro F1" if TASK == "classification" else "validation minus RMSE")
    plt.tight_layout(); plt.savefig(out_dir / "training_curves.png", dpi=150); plt.show()

model.load_state_dict(torch.load(ckpt_path, map_location=device)["state_dict"])
print("loaded the best weights from", ckpt_path)

## Step 6: final test evaluation (run once, at the end)

Compared with a reference point computed from the training split: the majority class, or the training mean.

In [ ]:
_, test_out, test_y = run_epoch(test_loader, False)
if TASK == "classification":
    pred, true = test_out.argmax(1).numpy(), test_y.numpy()
    base = np.full_like(true, np.bincount([class_to_idx[c] for c in train_df["label"]], minlength=n_out).argmax())
    results = {"model": {"accuracy": accuracy_score(true, pred), "macro_f1": f1_score(true, pred, average="macro")},
               "majority_baseline": {"accuracy": accuracy_score(true, base), "macro_f1": f1_score(true, base, average="macro")}}
    print(pd.DataFrame(results).T.round(3))
    fig, ax = plt.subplots(figsize=(5, 4.6))
    ConfusionMatrixDisplay(confusion_matrix(true, pred, labels=list(range(n_out))), display_labels=classes).plot(
        ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title("Test confusion matrix"); plt.tight_layout(); plt.savefig(out_dir / "confusion_matrix.png", dpi=150); plt.show()
else:
    pred, true = test_out.numpy() * y_std + y_mean, test_y.numpy() * y_std + y_mean      # original units
    base = np.broadcast_to(y_mean, true.shape)
    rmse = lambda a, b: float(np.sqrt(((a - b) ** 2).mean()))
    results = {"model": {"RMSE": rmse(pred, true), "MAE": float(np.abs(pred - true).mean())},
               "mean_baseline": {"RMSE": rmse(base, true), "MAE": float(np.abs(base - true).mean())}}
    print(pd.DataFrame(results).T.round(3))
    fig, ax = plt.subplots(figsize=(4.6, 4.6))
    ax.scatter(true[:, 0], pred[:, 0], s=10)
    lims = [float(true[:, 0].min()), float(true[:, 0].max())]; ax.plot(lims, lims, "k--", lw=1, label="y = x")
    ax.set_xlabel("true (original units)"); ax.set_ylabel("predicted (original units)"); ax.legend()
    ax.set_title("Test: predicted vs true"); plt.tight_layout(); plt.savefig(out_dir / "pred_vs_true.png", dpi=150); plt.show()
json.dump(results, open(out_dir / "results.json", "w"), indent=2, default=float)

## Step 7: failure cases

The most confident mistakes (classification) or the largest errors (regression). Pick one to explain in the video.

In [ ]:
if TASK == "classification":
    prob = torch.softmax(test_out, 1).numpy(); wrong = np.where(pred != true)[0]
    order = wrong[np.argsort(-prob[wrong].max(1))][:6]
    titles = [f"true {classes[true[i]]}\npred {classes[pred[i]]} ({prob[i].max():.0%})" for i in order]
else:
    order = np.argsort(-np.abs(pred - true).sum(1))[:6]
    titles = [f"true {true[i, 0]:.2f}\npred {pred[i, 0]:.2f}" for i in order]
if len(order) == 0:
    print("No test errors: show the least confident correct predictions instead.")
else:
    fig, axes = plt.subplots(1, len(order), figsize=(2.4 * len(order), 2.9), squeeze=False)
    for ax, i, t in zip(axes[0], order, titles):
        ax.imshow(Image.open(test_df.iloc[i]["paths"][-1]).convert("RGB")); ax.set_title(t, fontsize=8); ax.axis("off")
    plt.suptitle("Failure cases (test split)"); plt.tight_layout(); plt.savefig(out_dir / "failure_cases.png", dpi=150); plt.show()

## Step 8: demo clip for the video

An animated GIF of the model predicting on test samples (green = correct, red = wrong). A screen recording of this cell works too.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
idx = np.random.default_rng(SEED).choice(len(test_df), size=min(12, len(test_df)), replace=False)
fig, ax = plt.subplots(figsize=(4, 4.4))
def draw(k):
    i = idx[k]; ax.clear(); ax.axis("off")
    ax.imshow(Image.open(test_df.iloc[i]["paths"][-1]).convert("RGB"))
    if TASK == "classification":
        ax.set_title(f"pred: {classes[pred[i]]}\ntrue: {classes[true[i]]}", fontsize=10,
                     color="green" if pred[i] == true[i] else "red")
    else:
        ax.set_title(f"pred: {pred[i, 0]:.2f}   true: {true[i, 0]:.2f}", fontsize=10)
FuncAnimation(fig, draw, frames=len(idx), interval=900).save(out_dir / "demo.gif", writer=PillowWriter(fps=1))
plt.close(fig)
print("saved", out_dir / "demo.gif")

In [ ]:
summary = {"task": TASK, "model": MODEL_NAME, "frames_per_sample": N_FRAMES, "image_size": IMG_SIZE,
           "split_sizes": {s: len(d) for s, d in splits}, "results": results}
print(json.dumps(summary, indent=2, default=float))
print("Figures and files for the video team:", sorted(p.name for p in out_dir.iterdir()))

## Before the code zip

- [ ] `DEMO_MODE = False`, `TRAIN = False`, and *Restart and run all* works top to bottom
- [ ] `best_model.pt` is included; raw data is not
- [ ] The README says which dataset subset we used, how we split it, and where to download it
- [ ] Every pretrained weight, dataset and adapted code snippet is cited